In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_raw=pd.read_csv("../data/raw/ecommerce_retail_transactions_raw.csv")

In [ ]:
df_raw.shape

(12180, 13)

In [ ]:
df=df_raw.copy()

In [ ]:
baseline_rows = len(df)
baseline_columns = len(df.columns)

print("Rows:", baseline_rows)
print("Columns:", baseline_columns)

Rows: 12180
Columns: 13


In [ ]:
print(df.columns.tolist())

['Order_ID', 'Customer_ID', 'Order_Date', 'Product_Category', 'Product_Name', 'Quantity', 'Unit_Price_USD', 'Discount_Percent', 'Payment_Method', 'Shipping_City', 'Country', 'Order_Status', 'Customer_Rating']


In [ ]:
df["Order_Date"].head(10)

0     2026-04-03
1     10-15-2024
2    17 Jun 2024
3     16/10/2025
4    06 Apr 2026
5     11-09-2024
6     21/03/2026
7     10/07/2024
8     19/08/2025
9     07-14-2025
Name: Order_Date, dtype: object

In [ ]:
df["Order_Date_Raw"]=df["Order_Date"]

In [ ]:
parsed_dates=pd.to_datetime(
    df["Order_Date"],
    errors="coerce"
)

In [ ]:
parsed_dates.isna().sum()

np.int64(9692)

In [ ]:
df.loc[
    parsed_dates.isna(),
    ["Order_Date_Raw"]
].head(20)

,Order_Date_Raw
1,10-15-2024
2,17 Jun 2024
3,16/10/2025
4,06 Apr 2026
5,11-09-2024
6,21/03/2026
7,10/07/2024
8,19/08/2025
9,07-14-2025
10,06-05-2024


In [ ]:
df[parsed_dates.notna()]

,Order_ID,Customer_ID,Order_Date,Product_Category,Product_Name,Quantity,Unit_Price_USD,Discount_Percent,Payment_Method,Shipping_City,Country,Order_Status,Customer_Rating,Order_Date_Raw
0,ORD011416,CUST00003,2026-04-03,Books,Self-Help Book,3,9.16,5.0,Net Banking,London,UK,Delivered,5.0,2026-04-03
26,ORD011023,CUST01851,2025-04-10,Electronics,Wireless Earbuds,4,70.78,15.0,CREDIT CARD,Melbourne,australia,Delivered,5.0,2025-04-10
28,ORD007093,CUST00950,2024-02-20,Home & Kitchen,Blender,1,168.65,0.0,net banking,Chicago,usa,Pending,NaN,2024-02-20
30,ORD011670,CUST01862,2024-11-17,Electronics,Wireless Earbuds,2,204.37,15.0,Debit Card,Abu Dhabi,U.A.E,Cancelled,NaN,2024-11-17
34,ORD003182,CUST02431,2024-11-06,Beauty,Sunscreen SPF50,1,58.51,0.0,COD,NaN,UK,Shipped,NaN,2024-11-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12163,ORD008317,CUST00541,2024-07-19,Sports,Cycling Helmet,1,107.30,10.0,net banking,Berlin,DE,Delivered,3.0,2024-07-19
12167,ORD005168,CUST02663,2025-10-06,Grocery,Green Tea Pack,2,6.95,10.0,Pay Pal,Melbourne,australia,Delivered,4.0,2025-10-06
12173,ORD011528,CUST00830,2025-07-04,Home & Kitchen,Blender,1,38.54,5.0,DEBIT CARD,Toronto,canada,Delivered,5.0,2025-07-04
12175,ORD002515,CUST00071,2025-10-17,Home & Kitchen,Storage Organizer,1,106.25,0.0,net banking,Chicago,U.S.A,Delivered,4.0,2025-10-17


In [ ]:
clean_dates=pd.Series(
    pd.NaT,
    index=df.index,
    dtype="datetime64[ns]"
)

In [ ]:
raw = df["Order_Date_Raw"].astype("string").str.strip()

mask_dash = raw.str.match(
    r"^\d{1,2}-\d{1,2}-\d{4}$",
    na=False
)

clean_dates.loc[mask_dash] = pd.to_datetime(
    raw.loc[mask_dash],
    format="%m-%d-%Y",
    errors="coerce"
)

In [ ]:
mask_slash = raw.str.match(
    r"^\d{1,2}/\d{1,2}/\d{4}$",
    na=False
)

clean_dates.loc[mask_slash] = pd.to_datetime(
    raw.loc[mask_slash],
    format="%d/%m/%Y",
    errors="coerce"
)

In [ ]:
mask_text_month = raw.str.match(
    r"^\d{1,2} [A-Za-z]{3} \d{4}$",
    na=False
)

clean_dates.loc[mask_text_month] = pd.to_datetime(
    raw.loc[mask_text_month],
    format="%d %b %Y",
    errors="coerce"
)

In [ ]:
mask_year_slash = raw.str.match(
    r"^\d{4}/\d{1,2}/\d{1,2}$",
    na=False
)

clean_dates.loc[mask_year_slash] = pd.to_datetime(
    raw.loc[mask_year_slash],
    format="%Y/%m/%d",
    errors="coerce"
)

In [ ]:
mask_year_dash = raw.str.match(
    r"^\d{4}-\d{1,2}-\d{1,2}$",
    na=False
)

clean_dates.loc[mask_year_dash] = pd.to_datetime(
    raw.loc[mask_year_dash],
    format="%Y-%m-%d",
    errors="coerce"
)

In [ ]:
df["Order_Date"]=clean_dates

In [ ]:
print("Original rows:", len(df))
print("Missing raw dates:", df["Order_Date_Raw"].isna().sum())
print("Failed after parsing:", df["Order_Date"].isna().sum())

Original rows: 12180
Missing raw dates: 0
Failed after parsing: 0


In [ ]:
df["Order_Date"].dtype

dtype('<M8[ns]')

In [ ]:
df["Order_Date"].head(10)

0   2026-04-03
1   2024-10-15
2   2024-06-17
3   2025-10-16
4   2026-04-06
5   2024-11-09
6   2026-03-21
7   2024-07-10
8   2025-08-19
9   2025-07-14
Name: Order_Date, dtype: datetime64[ns]

In [ ]:
df.loc[
    df["Order_Date"].isna(),
    ["Order_Date_Raw"]
].head(50)

,Order_Date_Raw


In [ ]:
print("Minimum date:", df["Order_Date"].min())
print("Maximum date:", df["Order_Date"].max())

Minimum date: 2024-01-01 00:00:00
Maximum date: 2026-06-30 00:00:00


In [ ]:
df["Order_Date"] = df["Order_Date"].dt.strftime("%d-%m-%Y")

In [ ]:
df["Order_Date"].head(20)

0     03-04-2026
1     15-10-2024
2     17-06-2024
3     16-10-2025
4     06-04-2026
5     09-11-2024
6     21-03-2026
7     10-07-2024
8     19-08-2025
9     14-07-2025
10    05-06-2024
11    09-06-2025
12    11-06-2025
13    09-12-2024
14    23-12-2025
15    15-07-2025
16    05-11-2025
17    11-11-2025
18    04-04-2025
19    16-04-2026
Name: Order_Date, dtype: object

In [ ]:
df["Discount_Percent"].isna().sum()

np.int64(735)

In [ ]:
df[df["Discount_Percent"].isna()][
    [
        "Order_ID",
        "Product_Category",
        "Product_Name",
        "Quantity",
        "Unit_Price_USD",
        "Payment_Method",
        "Order_Status"
    ]
].head(20)

,Order_ID,Product_Category,Product_Name,Quantity,Unit_Price_USD,Payment_Method,Order_Status
4,ORD000740,Home & Kitchen,Non-Stick Pan,2,72.00,Credit_Card,Delivered
7,ORD011407,Books,Cookbook,3,37.49,Cash on Delivery,Returned
8,ORD005427,Home & Kitchen,Vacuum Cleaner,4,21.44,NetBanking,Shipped
19,ORD011697,Fashion,Cotton T-Shirt,3,86.68,Pay Pal,Shipped
21,ORD009446,Sports,Football,1,53.17,upi,Delivered
56,ORD002014,Home & Kitchen,Vacuum Cleaner,1,106.12,debit card,Delivered
92,ORD006646,Sports,Dumbbell Set,2,119.72,upi,Returned
108,ORD000283,Beauty,Face Moisturizer,1,9.40,UPI,Delivered
117,ORD005264,Beauty,Sunscreen SPF50,3,18.00,net banking,Delivered
121,ORD008722,Electronics,Power Bank,4,187.09,DEBIT CARD,Shipped


In [ ]:
df[df["Discount_Percent"] == 0][
    ["Order_ID", "Product_Name", "Discount_Percent"]
].head(20)

,Order_ID,Product_Name,Discount_Percent
3,ORD006437,Hair Dryer,0.0
5,ORD011216,Storage Organizer,0.0
12,ORD006143,Remote Control Car,0.0
13,ORD002406,Action Figure,0.0
14,ORD008099,Cycling Helmet,0.0
15,ORD000325,Webcam,0.0
20,ORD003641,Cricket Bat,0.0
22,ORD006908,Gaming Mouse,0.0
23,ORD001501,Cookbook,0.0
28,ORD007093,Blender,0.0


In [ ]:
print(
    "Zero discounts:",
    (df["Discount_Percent"] == 0).sum()
)

print(
    "Missing discounts:",
    df["Discount_Percent"].isna().sum()
)

Zero discounts: 4099
Missing discounts: 735


In [ ]:
pd.crosstab(
    df["Order_Status"],
    df["Discount_Percent"].isna(),
    normalize="index"
).round(3)

Discount_Percent,False,True
Order_Status,,
Cancelled,0.948,0.052
Delivered,0.939,0.061
Pending,0.949,0.051
Returned,0.939,0.061
Shipped,0.929,0.071


In [ ]:
pd.crosstab(
    df["Product_Category"],
    df["Discount_Percent"].isna(),
    normalize="index"
).round(3)

Discount_Percent,False,True
Product_Category,,
Beauty,0.925,0.075
Books,0.950,0.050
Electronics,0.931,0.069
Fashion,0.942,0.058
Grocery,0.943,0.057
Home & Kitchen,0.941,0.059
Sports,0.944,0.056
Toys,0.941,0.059


In [ ]:
df["Discount_Percent"] = df["Discount_Percent"].fillna(0)

In [ ]:
print(
    "Missing discounts:",
    df["Discount_Percent"].isna().sum()
)

Missing discounts: 0


In [ ]:
df["Discount_Percent"].describe()

count    12180.000000
mean         6.858785
std          7.117463
min          0.000000
25%          0.000000
50%          5.000000
75%         10.000000
max         25.000000
Name: Discount_Percent, dtype: float64

In [ ]:
df[
    (df["Discount_Percent"] < 0) |
    (df["Discount_Percent"] > 100)
][["Order_ID", "Discount_Percent"]].head(20)

,Order_ID,Discount_Percent


In [ ]:
df["Discount_Percent"].value_counts().sort_index()

Discount_Percent
0.0     4834
5.0     2265
10.0    2281
15.0    1640
20.0     839
25.0     321
Name: count, dtype: int64

In [ ]:
df["Shipping_City"].isna().sum()

np.int64(502)

In [ ]:
df[df["Shipping_City"].isna()][
    [
        "Order_ID",
        "Customer_ID",
        "Country",
        "Order_Status",
        "Product_Category"
    ]
].head(20)

,Order_ID,Customer_ID,Country,Order_Status,Product_Category
34,ORD003182,CUST02431,UK,Shipped,Beauty
42,ORD001098,CUST02792,United States,Delivered,Books
56,ORD002014,CUST00395,USA,Delivered,Home & Kitchen
132,ORD006138,CUST01368,canada,Delivered,Beauty
164,ORD004007,CUST02743,Canada,Delivered,Beauty
186,ORD010738,CUST01848,Germany,Shipped,Grocery
208,ORD000339,CUST00893,Australia,Delivered,Beauty
266,ORD006280,CUST02064,uae,Shipped,Toys
273,ORD000856,CUST02332,United States,Cancelled,Sports
343,ORD007614,CUST00392,USA,Delivered,Fashion


In [ ]:
pd.crosstab(
    df["Order_Status"],
    df["Shipping_City"].isna(),
    normalize="index"
).round(3)

Shipping_City,False,True
Order_Status,,
Cancelled,0.963,0.037
Delivered,0.959,0.041
Pending,0.952,0.048
Returned,0.967,0.033
Shipped,0.957,0.043


In [ ]:
df[
    df["Shipping_City"].isna()
][
    ["Shipping_City", "Country", "Order_Status"]
].head(20)

,Shipping_City,Country,Order_Status
34,NaN,UK,Shipped
42,NaN,United States,Delivered
56,NaN,USA,Delivered
132,NaN,canada,Delivered
164,NaN,Canada,Delivered
186,NaN,Germany,Shipped
208,NaN,Australia,Delivered
266,NaN,uae,Shipped
273,NaN,United States,Cancelled
343,NaN,USA,Delivered


In [ ]:
df["Shipping_City"] = df["Shipping_City"].fillna("Unknown")

In [ ]:
df["Shipping_City"].isna().sum()

np.int64(0)

In [ ]:
df["Shipping_City"].value_counts().head(30)

Shipping_City
Abu Dhabi      881
Dubai          808
Hamburg        609
Melbourne      586
Manchester     583
Brisbane       579
Birmingham     571
London         563
Munich         558
Toronto        545
Montreal       540
Berlin         535
Sydney         512
Unknown        502
Vancouver      489
New York       456
Hyderabad      433
Chicago        425
Bengaluru      414
Los Angeles    401
Delhi          400
Mumbai         399
Houston        391
Name: count, dtype: int64

In [ ]:
df["Customer_Rating"].isna().sum()  

np.int64(6150)

In [ ]:
pd.crosstab(
    df["Order_Status"],
    df["Customer_Rating"].isna(),
    normalize="index"
).round(3)

Customer_Rating,False,True
Order_Status,,
Cancelled,0.000,1.000
Delivered,0.904,0.096
Pending,0.000,1.000
Returned,0.000,1.000
Shipped,0.000,1.000


In [ ]:
invalid_ratings = df[
    (df["Customer_Rating"] < 1) |
    (df["Customer_Rating"] > 5)
]

print("Invalid ratings:", len(invalid_ratings))

Invalid ratings: 0


In [ ]:
rating_response_rate = (
    df["Customer_Rating"].notna().mean() * 100
)

print(f"Rating response rate: {rating_response_rate:.2f}%")

Rating response rate: 49.51%


In [ ]:
average_rating = df["Customer_Rating"].mean()

print(f"Average rating: {average_rating:.2f}")

Average rating: 3.90


In [ ]:
duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 180


In [ ]:
df[df.duplicated(keep=False)].sort_values(
    "Order_ID"
).head(20)

,Order_ID,Customer_ID,Order_Date,Product_Category,Product_Name,Quantity,Unit_Price_USD,Discount_Percent,Payment_Method,Shipping_City,Country,Order_Status,Customer_Rating,Order_Date_Raw
10813,ORD000018,CUST00794,20-07-2024,Books,Kids Storybook,2,23.28,15.0,DEBIT CARD,Bengaluru,India,Shipped,NaN,07-20-2024
9400,ORD000018,CUST00794,20-07-2024,Books,Kids Storybook,2,23.28,15.0,DEBIT CARD,Bengaluru,India,Shipped,NaN,07-20-2024
4476,ORD000036,CUST00063,04-05-2026,Electronics,Power Bank,1,31.76,5.0,Pay Pal,Toronto,CA,Delivered,4.0,04 May 2026
6580,ORD000036,CUST00063,04-05-2026,Electronics,Power Bank,1,31.76,5.0,Pay Pal,Toronto,CA,Delivered,4.0,04 May 2026
7763,ORD000136,CUST00688,27-09-2024,Fashion,Sunglasses,2,54.91,15.0,net banking,Sydney,australia,Delivered,4.0,27 Sep 2024
952,ORD000136,CUST00688,27-09-2024,Fashion,Sunglasses,2,54.91,15.0,net banking,Sydney,australia,Delivered,4.0,27 Sep 2024
881,ORD000283,CUST02279,04-05-2026,Beauty,Face Moisturizer,1,9.40,0.0,UPI,Manchester,uk,Delivered,4.0,04 May 2026
108,ORD000283,CUST02279,04-05-2026,Beauty,Face Moisturizer,1,9.40,0.0,UPI,Manchester,uk,Delivered,4.0,04 May 2026
2941,ORD000385,CUST02389,29-08-2025,Fashion,Hoodie,2,13.65,5.0,COD,Brisbane,AU,Delivered,4.0,08-29-2025
4518,ORD000385,CUST02389,29-08-2025,Fashion,Hoodie,2,13.65,5.0,COD,Brisbane,AU,Delivered,4.0,08-29-2025


In [ ]:
df = df.drop_duplicates()

In [ ]:
print("Rows after removing exact duplicates:", len(df))

Rows after removing exact duplicates: 12000


In [ ]:
print("Remaining exact duplicates:", df.duplicated().sum())

Remaining exact duplicates: 0


In [ ]:
df["Payment_Method"].value_counts(dropna=False)

Payment_Method
paypal              701
net banking         693
PayPal              689
Net Banking         682
UPI                 678
DEBIT CARD          671
NetBanking          650
Pay Pal             643
U.P.I               638
debit card          623
Debit Card          616
upi                 615
COD                 534
CREDIT CARD         529
cash on delivery    520
cod                 515
Credit_Card         506
Cash on Delivery    501
credit card         499
Credit Card         497
Name: count, dtype: int64

In [ ]:
df["Payment_Method"] = (
    df["Payment_Method"]
    .astype("string")
    .str.strip()
)

In [ ]:
payment_mapping={
    "paypal":"PayPal",
    "net banking":"Net Banking",
    "DEBIT CARD":"Debit Card",
    "NetBanking":"Net Banking",
    "U.P.I":"UPI",
    "debit card":"Debit Card",
    "upi":"UPI",
    "COD":"Cash on Delivery",
    "CREDIT CARD":"Credit Card",
    "cash on delivery":"Cash on Delivery",
    "Credit_Card":"Credit Card",
    "credit card":"Credit Card",
    "Pay Pal":"PayPal",
    "cod":"Cash on Delivery",
}

In [ ]:
df["Payment_Method"]=(
    df["Payment_Method"]
    .replace(payment_mapping)
)

In [ ]:
df["Payment_Method"].value_counts(dropna=False)

Payment_Method
Cash on Delivery    2070
PayPal              2033
Credit Card         2031
Net Banking         2025
UPI                 1931
Debit Card          1910
Name: count, dtype: Int64

In [ ]:
df["Country"].value_counts(dropna=False)

Country
UAE               593
australia         591
DE                587
Germany           581
germany           579
uae               577
india             576
Australia         576
IN                575
U.A.E             572
canada            563
AU                540
CA                539
India             539
Canada            533
uk                455
U.K.              444
United Kingdom    439
UK                430
usa               366
USA               361
United States     341
US                335
U.S.A             308
Name: count, dtype: int64

In [ ]:
df["Country"] = (
    df["Country"]
    .astype("string")
    .str.strip()
)

In [ ]:
df["Country"] = (
    df["Country"]
    .astype("string")
    .str.lower()
)

In [ ]:
df["Country"].value_counts(dropna=False)

Country
uae               1170
australia         1167
germany           1160
india             1115
canada            1096
uk                 885
usa                727
de                 587
in                 575
u.a.e              572
au                 540
ca                 539
u.k.               444
united kingdom     439
united states      341
us                 335
u.s.a              308
Name: count, dtype: Int64

In [ ]:
country_mapping={
    "australia":"Australia",
    "germany":"Germany",
    "india":"India",
    "canada":"Canada",
    "uk":"United Kingdom",
    "united kingdom":"United Kingdom",
    "au":"Australia",
    "de":"Germany",
    "in":"India",
    "ca":"Canada",
    "united states":"United States",
    "us":"United States",
    "u.k.":"United Kingdom",
    "usa":"United States",
    "u.s.a":"United States",
    "uae":"United Arab Emirates",
    "u.a.e":"United Arab Emirates"
}


In [ ]:
df["Country"]=df["Country"].replace(country_mapping)

In [ ]:
df["Country"].value_counts(dropna=False)

Country
United Kingdom          1768
Germany                 1747
United Arab Emirates    1742
United States           1711
Australia               1707
India                   1690
Canada                  1635
Name: count, dtype: Int64

In [ ]:
df["Order_Status"].value_counts(dropna=False)

Order_Status
Delivered    6578
Shipped      1816
Pending      1395
Cancelled    1196
Returned     1015
Name: count, dtype: int64

In [ ]:
df["Order_Status"] = (
    df["Order_Status"]
    .astype("string")
    .str.strip()
)

In [ ]:
df["Product_Category"].value_counts(dropna=False)

Product_Category
Home & Kitchen    1545
Toys              1536
Books             1526
Fashion           1515
Beauty            1484
Electronics       1483
Grocery           1472
Sports            1439
Name: count, dtype: int64

In [ ]:
df["Product_Category"] = (
    df["Product_Category"]
    .astype("string")
    .str.strip()
)

In [ ]:
df["Shipping_City"].value_counts(dropna=False)

Shipping_City
Abu Dhabi      867
Dubai          791
Hamburg        601
Melbourne      578
Manchester     577
Brisbane       570
Birmingham     564
London         560
Munich         552
Toronto        536
Montreal       530
Berlin         521
Sydney         504
Unknown        494
Vancouver      487
New York       448
Hyderabad      426
Chicago        423
Bengaluru      406
Los Angeles    398
Delhi          393
Mumbai         392
Houston        382
Name: count, dtype: int64

In [ ]:
df["Shipping_City"] = (
    df["Shipping_City"]
    .astype("string")
    .str.strip()
)

In [ ]:
df["Product_Name"].value_counts(dropna=False)

Product_Name
Fiction Novel          403
Puzzle 1000pc          392
Cookbook               388
Remote Control Car     384
Action Figure          383
Green Tea Pack         378
Building Blocks Set    377
Kids Storybook         373
Olive Oil 1L           370
Basmati Rice 5kg       366
Self-Help Book         362
Almonds 500g           358
Dumbbell Set           293
Cricket Bat            292
Football               291
Cycling Helmet         290
Yoga Mat               273
Perfume                273
Shampoo                256
Lipstick               255
Air Fryer              242
Sunscreen SPF50        240
Hair Dryer             239
LED Desk Lamp          237
Non-Stick Pan          232
Storage Organizer      230
Face Moisturizer       221
Ceramic Mug Set        218
Sunglasses             214
Power Bank             213
Leather Wallet         210
Running Shoes          200
Smartwatch             196
Vacuum Cleaner         196
Formal Shirt           194
Blender                190
Bluetooth Speak

In [ ]:
df["Product_Name"] = (
    df["Product_Name"]
    .astype("string")
    .str.strip()
)

In [ ]:
print("PAYMENT METHODS")
print(df["Payment_Method"].value_counts(dropna=False))

print("\nCOUNTRIES")
print(df["Country"].value_counts(dropna=False))

print("\nORDER STATUS")
print(df["Order_Status"].value_counts(dropna=False))

print("\nPRODUCT CATEGORIES")
print(df["Product_Category"].value_counts(dropna=False))

PAYMENT METHODS
Payment_Method
Cash on Delivery    2070
PayPal              2033
Credit Card         2031
Net Banking         2025
UPI                 1931
Debit Card          1910
Name: count, dtype: Int64

COUNTRIES
Country
United Kingdom          1768
Germany                 1747
United Arab Emirates    1742
United States           1711
Australia               1707
India                   1690
Canada                  1635
Name: count, dtype: Int64

ORDER STATUS
Order_Status
Delivered    6578
Shipped      1816
Pending      1395
Cancelled    1196
Returned     1015
Name: count, dtype: Int64

PRODUCT CATEGORIES
Product_Category
Home & Kitchen    1545
Toys              1536
Books             1526
Fashion           1515
Beauty            1484
Electronics       1483
Grocery           1472
Sports            1439
Name: count, dtype: Int64


In [ ]:
print("\nSHIPPING CITIES")
print(df["Shipping_City"].value_counts(dropna=False).head(30))


SHIPPING CITIES
Shipping_City
Abu Dhabi      867
Dubai          791
Hamburg        601
Melbourne      578
Manchester     577
Brisbane       570
Birmingham     564
London         560
Munich         552
Toronto        536
Montreal       530
Berlin         521
Sydney         504
Unknown        494
Vancouver      487
New York       448
Hyderabad      426
Chicago        423
Bengaluru      406
Los Angeles    398
Delhi          393
Mumbai         392
Houston        382
Name: count, dtype: Int64


In [ ]:
categorical_cols = [
    "Payment_Method",
    "Country",
    "Order_Status",
    "Product_Category",
    "Shipping_City",
    "Product_Name"
]

In [ ]:
df[categorical_cols].isna().sum()

Payment_Method      0
Country             0
Order_Status        0
Product_Category    0
Shipping_City       0
Product_Name        0
dtype: int64

In [ ]:
print("Quantity <= 0:")
print(
    df[df["Quantity"] <= 0][
        ["Order_ID", "Product_Name", "Quantity", "Order_Status"]
    ]
)

print("\nCount:", (df["Quantity"] <= 0).sum())

Quantity <= 0:
        Order_ID         Product_Name  Quantity Order_Status
197    ORD006541        Running Shoes        -1     Returned
265    ORD007225         Dumbbell Set         0    Delivered
285    ORD000653           Sunglasses        -1    Delivered
339    ORD008111             Yoga Mat         0    Delivered
378    ORD010769           Smartwatch        -1    Delivered
...          ...                  ...       ...          ...
11819  ORD007397              Shampoo        -1      Pending
11905  ORD003542       Green Tea Pack         0      Shipped
11965  ORD009023              Blender         0    Delivered
11990  ORD006503  Building Blocks Set        -1    Delivered
12095  ORD000610             Cookbook         0    Delivered

[186 rows x 4 columns]

Count: 186


In [ ]:
quantity_invalid = df[df["Quantity"] <= 0]

print("Invalid quantity records:", len(quantity_invalid))

Invalid quantity records: 186


In [ ]:
print("Quantity <= 0:",
      (df["Quantity"] <= 0).sum())

print("Price <= 0:",
      (df["Unit_Price_USD"] <= 0).sum())

print("Discount outside 0–100:",
      (
          (df["Discount_Percent"] < 0) |
          (df["Discount_Percent"] > 100)
      ).sum())

print("Rating outside 1–5:",
      (
          (df["Customer_Rating"] < 1) |
          (df["Customer_Rating"] > 5)
      ).sum())

Quantity <= 0: 186
Price <= 0: 0
Discount outside 0–100: 0
Rating outside 1–5: 0


In [ ]:
pd.crosstab(
    df["Order_Status"],
    df["Quantity"] <= 0,
    normalize="index"
).round(3)

Quantity,False,True
Order_Status,,
Cancelled,0.985,0.015
Delivered,0.984,0.016
Pending,0.982,0.018
Returned,0.982,0.018
Shipped,0.990,0.010


In [ ]:
print("Zero quantity:",
      (df["Quantity"] == 0).sum())

print("Negative quantity:",

      (df["Quantity"] < 0).sum())

Zero quantity: 90
Negative quantity: 96


In [ ]:
df.loc[
    df["Quantity"] <= 0,
    "Quantity"
].value_counts().sort_index()

Quantity
-1    96
 0    90
Name: count, dtype: int64

In [ ]:
pd.crosstab(
    df["Order_Status"],
    df["Quantity"] == 0,
    normalize="index"   
)

Quantity,False,True
Order_Status,,
Cancelled,0.993311,0.006689
Delivered,0.991487,0.008513
Pending,0.992115,0.007885
Returned,0.991133,0.008867
Shipped,0.996696,0.003304


In [ ]:
invalid_quantity = df[df["Quantity"] <= 0]

invalid_quantity[
    [
        "Order_ID",
        "Customer_ID",
        "Order_Date",
        "Product_Category",
        "Product_Name",
        "Quantity",
        "Unit_Price_USD",
        "Discount_Percent",
        "Payment_Method",
        "Order_Status"
    ]
]

,Order_ID,Customer_ID,Order_Date,Product_Category,Product_Name,Quantity,Unit_Price_USD,Discount_Percent,Payment_Method,Order_Status
197,ORD006541,CUST02024,08-11-2024,Fashion,Running Shoes,-1,79.88,15.0,Credit Card,Returned
265,ORD007225,CUST00439,12-03-2025,Sports,Dumbbell Set,0,27.32,5.0,UPI,Delivered
285,ORD000653,CUST03243,28-11-2024,Fashion,Sunglasses,-1,83.03,0.0,PayPal,Delivered
339,ORD008111,CUST01212,06-07-2024,Sports,Yoga Mat,0,99.03,0.0,Net Banking,Delivered
378,ORD010769,CUST00449,03-07-2025,Electronics,Smartwatch,-1,186.96,20.0,Debit Card,Delivered
...,...,...,...,...,...,...,...,...,...,...
11819,ORD007397,CUST00764,30-04-2024,Beauty,Shampoo,-1,24.49,10.0,Cash on Delivery,Pending
11905,ORD003542,CUST00215,11-01-2026,Grocery,Green Tea Pack,0,26.06,5.0,Net Banking,Shipped
11965,ORD009023,CUST02594,17-02-2024,Home & Kitchen,Blender,0,126.25,10.0,PayPal,Delivered
11990,ORD006503,CUST02234,10-11-2024,Toys,Building Blocks Set,-1,55.90,5.0,PayPal,Delivered


In [ ]:
invalid_quantity.groupby(
    ["Quantity", "Order_Status"]
    
).size()

Quantity  Order_Status
-1        Cancelled       10
          Delivered       50
          Pending         14
          Returned         9
          Shipped         13
 0        Cancelled        8
          Delivered       56
          Pending         11
          Returned         9
          Shipped          6
dtype: int64

In [ ]:
invalid_quantity.groupby("Quantity")["Unit_Price_USD"].describe()

,count,mean,std,min,25%,50%,75%,max
Quantity,,,,,,,,
-1,96.0,62.837187,54.008496,5.26,21.4275,46.055,83.9825,241.19
0,90.0,57.776444,48.862991,4.64,22.4725,37.455,81.2375,204.88


In [ ]:
invalid_quantity = df[df["Quantity"] <= 0].copy()

print("Records to remove:", len(invalid_quantity))

Records to remove: 186


In [ ]:
before_rows = len(df)

df = df[df["Quantity"] > 0].copy()

after_rows = len(df)

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before: 12000
Rows after: 11814
Rows removed: 186


In [ ]:
print(
    "Remaining invalid quantities:",
    (df["Quantity"] <= 0).sum()
)

Remaining invalid quantities: 0


In [ ]:
df["Quantity"].describe()

count    11814.000000
mean         2.058659
std          1.199634
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          5.000000
Name: Quantity, dtype: float64

In [ ]:
print("Current rows:", len(df))
print("Current columns:", len(df.columns))

Current rows: 11814
Current columns: 14


In [ ]:
df.columns


Index(['Order_ID', 'Customer_ID', 'Order_Date', 'Product_Category',
       'Product_Name', 'Quantity', 'Unit_Price_USD', 'Discount_Percent',
       'Payment_Method', 'Shipping_City', 'Country', 'Order_Status',
       'Customer_Rating', 'Order_Date_Raw'],
      dtype='object')

In [ ]:
df=df.drop(columns=["Order_Date_Raw"])

In [ ]:
df

,Order_ID,Customer_ID,Order_Date,Product_Category,Product_Name,Quantity,Unit_Price_USD,Discount_Percent,Payment_Method,Shipping_City,Country,Order_Status,Customer_Rating
0,ORD011416,CUST00003,03-04-2026,Books,Self-Help Book,3,9.16,5.0,Net Banking,London,United Kingdom,Delivered,5.0
1,ORD000974,CUST00269,15-10-2024,Books,Fiction Novel,1,34.24,5.0,Net Banking,Toronto,Canada,Shipped,NaN
2,ORD000617,CUST02075,17-06-2024,Home & Kitchen,Vacuum Cleaner,2,111.95,15.0,Debit Card,Brisbane,Australia,Shipped,NaN
3,ORD006437,CUST02109,16-10-2025,Beauty,Hair Dryer,1,31.84,0.0,Cash on Delivery,Birmingham,United Kingdom,Delivered,3.0
4,ORD000740,CUST00349,06-04-2026,Home & Kitchen,Non-Stick Pan,2,72.00,0.0,Credit Card,Montreal,Canada,Delivered,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12175,ORD002515,CUST00071,17-10-2025,Home & Kitchen,Storage Organizer,1,106.25,0.0,Net Banking,Chicago,United States,Delivered,4.0
12176,ORD011799,CUST02583,22-01-2024,Beauty,Sunscreen SPF50,1,24.76,15.0,PayPal,Mumbai,India,Delivered,5.0
12177,ORD006638,CUST02113,28-02-2024,Grocery,Olive Oil 1L,1,29.95,10.0,UPI,Melbourne,Australia,Pending,NaN
12178,ORD002576,CUST00298,01-06-2025,Beauty,Hair Dryer,1,41.41,15.0,Credit Card,Vancouver,Canada,Delivered,NaN


In [ ]:
print("Price <= 0:",
      (df["Unit_Price_USD"] <= 0).sum())

Price <= 0: 0


In [ ]:
df["Unit_Price_USD"].describe()

count    11814.000000
mean        61.542623
std         52.473860
min          3.010000
25%         22.032500
50%         42.800000
75%         87.737500
max        249.840000
Name: Unit_Price_USD, dtype: float64

In [ ]:
Q1 = df["Unit_Price_USD"].quantile(0.25)
Q3 = df["Unit_Price_USD"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Q1: 22.0325
Q3: 87.7375
IQR: 65.705
Lower bound: -76.525
Upper bound: 186.29500000000002


In [ ]:
price_outliers = df[
    df["Unit_Price_USD"] > upper_bound
].copy()

print("Potential price outliers:", len(price_outliers))

Potential price outliers: 415


In [ ]:
price_outliers[
    [
        "Order_ID",
        "Product_Category",
        "Product_Name",
        "Quantity",
        "Unit_Price_USD",
        "Discount_Percent",
        "Order_Status"
    ]
].head(30)

,Order_ID,Product_Category,Product_Name,Quantity,Unit_Price_USD,Discount_Percent,Order_Status
22,ORD006908,Electronics,Gaming Mouse,3,193.68,0.0,Delivered
30,ORD011670,Electronics,Wireless Earbuds,2,204.37,15.0,Cancelled
36,ORD005243,Electronics,Bluetooth Speaker,3,242.81,5.0,Delivered
49,ORD007126,Electronics,Webcam,1,188.64,0.0,Returned
50,ORD007171,Electronics,Webcam,3,189.65,0.0,Delivered
96,ORD009586,Electronics,Gaming Mouse,3,231.52,0.0,Pending
119,ORD004435,Electronics,Gaming Mouse,1,227.77,10.0,Delivered
121,ORD008722,Electronics,Power Bank,4,187.09,0.0,Shipped
134,ORD004497,Electronics,Power Bank,2,236.11,15.0,Delivered
259,ORD002844,Electronics,Wireless Earbuds,1,194.87,5.0,Shipped


In [ ]:
price_outliers["Product_Category"].value_counts()

Product_Category
Electronics    415
Name: count, dtype: Int64

In [ ]:
price_outliers.groupby(
    "Product_Category"
)["Unit_Price_USD"].describe()

,count,mean,std,min,25%,50%,75%,max
Product_Category,,,,,,,,
Electronics,415.0,217.845012,18.593375,186.3,201.14,217.47,233.795,249.84


In [ ]:
df["Gross_Sales_USD"] = (
    df["Quantity"] * df["Unit_Price_USD"]
)

In [ ]:
df[
    [
        "Quantity",
        "Unit_Price_USD",
        "Gross_Sales_USD"
    ]
].head(10)

,Quantity,Unit_Price_USD,Gross_Sales_USD
0,3,9.16,27.48
1,1,34.24,34.24
2,2,111.95,223.90
3,1,31.84,31.84
4,2,72.00,144.00
5,1,125.28,125.28
6,1,18.34,18.34
7,3,37.49,112.47
8,4,21.44,85.76
9,1,20.77,20.77


In [ ]:
df["Discount_Amount_USD"] = (
    df["Gross_Sales_USD"]
    * df["Discount_Percent"]
    / 100
)

In [ ]:
df["Net_Sales_USD"] = (
    df["Gross_Sales_USD"]
    - df["Discount_Amount_USD"]
    
)

In [ ]:
df[
    [
        "Quantity",
        "Unit_Price_USD",
        "Discount_Percent",
        "Gross_Sales_USD",
        "Discount_Amount_USD",
        "Net_Sales_USD"
    ]
].head(10)

,Quantity,Unit_Price_USD,Discount_Percent,Gross_Sales_USD,Discount_Amount_USD,Net_Sales_USD
0,3,9.16,5.0,27.48,1.374,26.106
1,1,34.24,5.0,34.24,1.712,32.528
2,2,111.95,15.0,223.90,33.585,190.315
3,1,31.84,0.0,31.84,0.000,31.840
4,2,72.00,0.0,144.00,0.000,144.000
5,1,125.28,0.0,125.28,0.000,125.280
6,1,18.34,5.0,18.34,0.917,17.423
7,3,37.49,0.0,112.47,0.000,112.470
8,4,21.44,0.0,85.76,0.000,85.760
9,1,20.77,10.0,20.77,2.077,18.693


In [ ]:
(df["Net_Sales_USD"] < 0).sum()

np.int64(0)

In [ ]:
(df["Net_Sales_USD"] == 0).sum()

np.int64(0)

In [ ]:
calculated_net = (
    df["Quantity"]
    * df["Unit_Price_USD"]
    * (1 - df["Discount_Percent"] / 100)
)

In [ ]:
difference = (
    df["Net_Sales_USD"] - calculated_net
).abs()

print("Maximum calculation difference:", difference.max())

Maximum calculation difference: 1.1368683772161603e-13


In [ ]:
df["Net_Sales_USD"].describe()

count    11814.000000
mean       118.336472
std        138.233784
min          2.552000
25%         31.642500
50%         70.416000
75%        147.606125
max       1249.200000
Name: Net_Sales_USD, dtype: float64

In [ ]:
df.nlargest(
    20,
    "Net_Sales_USD"
)[
    [
        "Order_ID",
        "Product_Name",
        "Product_Category",
        "Quantity",
        "Unit_Price_USD",
        "Discount_Percent",
        "Net_Sales_USD"
    ]
]

,Order_ID,Product_Name,Product_Category,Quantity,Unit_Price_USD,Discount_Percent,Net_Sales_USD
8491,ORD010505,USB-C Cable,Electronics,5,249.84,0.0,1249.2000
10734,ORD001754,Bluetooth Speaker,Electronics,5,247.40,0.0,1237.0000
4987,ORD010057,USB-C Cable,Electronics,5,245.47,0.0,1227.3500
939,ORD000043,Power Bank,Electronics,5,243.77,0.0,1218.8500
11618,ORD006310,Bluetooth Speaker,Electronics,5,241.36,0.0,1206.8000
4819,ORD007076,Wireless Earbuds,Electronics,5,240.80,0.0,1204.0000
9154,ORD007191,Gaming Mouse,Electronics,5,239.60,5.0,1138.1000
7131,ORD008635,Smartwatch,Electronics,5,249.81,10.0,1124.1450
10861,ORD010084,Bluetooth Speaker,Electronics,5,223.76,0.0,1118.8000
10250,ORD010385,Bluetooth Speaker,Electronics,5,230.12,10.0,1035.5400


In [ ]:
missing_price = df[df["Unit_Price_USD"].isna()]

print("Missing prices:", len(missing_price))

missing_price[
    [
        "Order_ID",
        "Product_Name",
        "Product_Category",
        "Quantity",
        "Discount_Percent",
        "Order_Status"
    ]
].head(30)

Missing prices: 0


,Order_ID,Product_Name,Product_Category,Quantity,Discount_Percent,Order_Status


In [ ]:
print("========== FINAL DATA QUALITY CHECK ==========")

print("\n1. ROWS & COLUMNS")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\n2. DUPLICATES")
print("Exact duplicates:", df.duplicated().sum())

print("\n3. ORDER DATE")
print("Missing dates:", df["Order_Date"].isna().sum())
print("Data type:", df["Order_Date"].dtype)
print("Min date:", df["Order_Date"].min())
print("Max date:", df["Order_Date"].max())

print("\n4. QUANTITY")
print("Missing:", df["Quantity"].isna().sum())
print("<= 0:", (df["Quantity"] <= 0).sum())

print("\n5. UNIT PRICE")
print("Missing:", df["Unit_Price_USD"].isna().sum())
print("<= 0:", (df["Unit_Price_USD"] <= 0).sum())

print("\n6. DISCOUNT")
print("Missing:", df["Discount_Percent"].isna().sum())
print(
    "Outside 0-100:",
    (
        (df["Discount_Percent"] < 0) |
        (df["Discount_Percent"] > 100)
    ).sum()
)

print("\n7. CUSTOMER RATING")
print("Missing:", df["Customer_Rating"].isna().sum())
print(
    "Outside 1-5:",
    (
        (df["Customer_Rating"] < 1) |
        (df["Customer_Rating"] > 5)
    ).sum()
)

print("\n8. SHIPPING CITY")
print("Missing:", df["Shipping_City"].isna().sum())

print("\n9. NET SALES")
print("Missing:", df["Net_Sales_USD"].isna().sum())
print("Negative:", (df["Net_Sales_USD"] < 0).sum())

print("\n10. CALCULATION VALIDATION")

calculated_net = (
    df["Quantity"]
    * df["Unit_Price_USD"]
    * (1 - df["Discount_Percent"] / 100)
)

difference = (
    df["Net_Sales_USD"] - calculated_net
).abs()

print("Maximum calculation difference:", difference.max())

========== FINAL DATA QUALITY CHECK ==========

1. ROWS & COLUMNS
Rows: 11814
Columns: 16

2. DUPLICATES
Exact duplicates: 0

3. ORDER DATE
Missing dates: 0
Data type: object
Min date: 01-01-2024
Max date: 31-12-2025

4. QUANTITY
Missing: 0
<= 0: 0

5. UNIT PRICE
Missing: 0
<= 0: 0

6. DISCOUNT
Missing: 0
Outside 0-100: 0

7. CUSTOMER RATING
Missing: 5968
Outside 1-5: 0

8. SHIPPING CITY
Missing: 0

9. NET SALES
Missing: 0
Negative: 0

10. CALCULATION VALIDATION
Maximum calculation difference: 1.1368683772161603e-13


In [ ]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

In [ ]:
print("Data type:", df["Order_Date"].dtype)
print("Missing dates:", df["Order_Date"].isna().sum())
print("Min date:", df["Order_Date"].min())
print("Max date:", df["Order_Date"].max())

Data type: datetime64[ns]
Missing dates: 0
Min date: 2024-01-01 00:00:00
Max date: 2026-06-30 00:00:00


In [ ]:
print(df["Order_Date"].dtype)

datetime64[ns]


In [ ]:
df.to_csv("../data/processed/ecommerce_retail_transactions_cleaned.csv", index=False)